In [ ]:
import pandas as pd
import json 


train_data = pd.read_csv('/path/to/data/train_set_one_verse_filtered.csv')
test_data = pd.read_csv('/path/to/data/test_data.csv')

In [8]:
# Display basic info about the datasets
print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
print("\nTrain data columns:", train_data.columns.tolist())
print("Test data columns:", test_data.columns.tolist())

# Show unique dataset names in each set
print("\nUnique dataset names in train data:", train_data['dataset_name'].unique())
print("Unique dataset names in test data:", test_data['dataset_name'].unique())

Train data shape: (427337, 31)
Test data shape: (6984, 30)

Train data columns: ['dataset_name', 'genre', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_url', 'rhyme', 'source', 'verses_explanation', 'poem_text_no_diacritics', 'poet_name_with_diacritics', 'keywords', 'key_phrases', 'corrupted_poem', 'corruption_assigned_template', 'corruption_type', 'corruption_metadata', 'norm_text', 'bucket']
Test data columns: ['corrupted_poem', 'corruption_assigned_template', 'corruption_metadata', 'corruption_type', 'dataset_name', 'genre', 'key_phrases', 'keywords', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text', 'poem_text_no_diacritics', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_name_with_diacriti

In [ ]:
def calculate_stats(df, set_name):
    """Calculate statistics grouped by dataset_name (or source for boda_scrapped)"""
    print(f"\n=== {set_name} Statistics ===")
    
    # Separate boda_scrapped data from others
    boda_data = df[df['dataset_name'] == 'boda_scrapped']
    other_data = df[df['dataset_name'] != 'boda_scrapped']
    
    results = []
    
    # Process non-boda_scrapped data grouped by dataset_name
    if not other_data.empty:
        grouped_others = other_data.groupby('dataset_name').agg({
            'poem_text': ['count', lambda x: x.str.len().mean()],
            'poem_verses': 'mean'
        }).round(2)
        grouped_others.columns = ['num_samples', 'avg_char_len', 'avg_verses']
        grouped_others = grouped_others.reset_index()
        grouped_others['group_type'] = 'dataset'
        results.append(grouped_others)
    
    # Process boda_scrapped data grouped by source
    if not boda_data.empty:
        grouped_boda = boda_data.groupby('source').agg({
            'poem_text': ['count', lambda x: x.str.len().mean()],
            'poem_verses': 'mean'
        }).round(2)
        grouped_boda.columns = ['num_samples', 'avg_char_len', 'avg_verses']
        grouped_boda = grouped_boda.reset_index()
        grouped_boda = grouped_boda.rename(columns={'source': 'dataset_name'})
        grouped_boda['dataset_name'] = 'boda_scrapped_' + grouped_boda['dataset_name'].astype(str)
        grouped_boda['group_type'] = 'source'
        results.append(grouped_boda)
    
    # Combine results
    if results:
        grouped = pd.concat(results, ignore_index=True)
        grouped = grouped[['dataset_name', 'num_samples', 'avg_char_len', 'avg_verses']]
    else:
        grouped = pd.DataFrame(columns=['dataset_name', 'num_samples', 'avg_char_len', 'avg_verses'])
    
    # Display results
    print(grouped.to_string(index=False))
    
    return grouped

# Calculate stats for train data
train_stats = calculate_stats(train_data, "TRAIN SET")

# Calculate total statistics for train split
print("\n=== TRAIN SET TOTAL STATISTICS ===")
total_train_stats = {
    'dataset_name': 'TOTAL_TRAIN',
    'num_samples': len(train_data),
    'avg_char_len': round(train_data['poem_text'].str.len().mean(), 2),
    'avg_verses': round(train_data['poem_verses'].mean(), 2)
}
print(f"Total samples: {total_train_stats['num_samples']}")
print(f"Average character length: {total_train_stats['avg_char_len']}")
print(f"Average number of verses: {total_train_stats['avg_verses']}")

# Calculate stats for test data  
test_stats = calculate_stats(test_data, "TEST SET")


=== TRAIN SET Statistics ===
            dataset_name  num_samples  avg_char_len  avg_verses
                AraPoems        62963       1039.51       22.01
   Arabic Poetry Dataset          662       1366.73       19.41
    Arabic-Poetry-Melody           48       1221.42       21.44
                  Ashaar       123581       1008.94       19.81
              adab_world            6       4971.33       93.33
                 arapoet         1303        734.90        9.25
                mawsooaa        18002        745.87       10.25
             poems_hakim            8       1198.38       24.88
      boda_scrapped_adab        70277       1014.66       35.33
     boda_scrapped_diwan        38005       1020.24       22.65
boda_scrapped_poets_gate       112482        806.69       15.58

=== TRAIN SET TOTAL STATISTICS ===
Total samples: 427337
Average character length: 950.87
Average number of verses: 21.39

=== TEST SET Statistics ===
dataset_name  num_samples  avg_char_len  avg_verse

: 

In [ ]:
# Create a combined summary table
print("\n=== COMBINED SUMMARY ===")

# Add total train stats to the train_stats dataframe
train_stats_with_total = pd.concat([
    train_stats,
    pd.DataFrame([total_train_stats])
], ignore_index=True)

combined_stats = pd.concat([
    train_stats_with_total.assign(set_type='Train'),
    test_stats.assign(set_type='Test')
])

# Pivot to show train/test side by side for each dataset
summary = combined_stats.pivot_table(
    index='dataset_name', 
    columns='set_type', 
    values=['num_samples', 'avg_char_len', 'avg_verses'], 
    fill_value=0
)

print(summary)


=== COMBINED SUMMARY ===
                         avg_char_len          avg_verses        num_samples  \
set_type                         Test    Train       Test  Train        Test   
dataset_name                                                                   
AraPoems                         0.00  1039.51       0.00  22.01         0.0   
Arabic Poetry Dataset            0.00  1366.73       0.00  19.41         0.0   
Arabic-Poetry-Melody             0.00  1221.42       0.00  21.44         0.0   
Ashaar                           0.00  1008.94       0.00  19.81         0.0   
FannOrFlop                    1420.45     0.00      17.97   0.00      6984.0   
adab_world                       0.00  4971.33       0.00  93.33         0.0   
arapoet                          0.00   734.90       0.00   9.25         0.0   
boda_scrapped_adab               0.00  1014.66       0.00  35.33         0.0   
boda_scrapped_diwan              0.00  1020.24       0.00  22.65         0.0   
boda_scrapped_

In [ ]:
# Optional: Save results to CSV files
train_stats_with_total.to_csv('train_dataset_stats.csv', index=False)
test_stats.to_csv('test_dataset_stats.csv', index=False)
combined_stats.to_csv('combined_dataset_stats.csv', index=False)

print("\nStatistics saved to CSV files:")
print("- train_dataset_stats.csv")
print("- test_dataset_stats.csv") 
print("- combined_dataset_stats.csv")